# Grain Boundary Misorientation Analysis: Educational Tutorial

Erik Bitzek, Tue Sep 30 18:35:57 CEST 2025

This notebook provides a step-by-step walkthrough of grain boundary misorientation calculations using quaternions and crystal symmetry.

## Learning Objectives

1. Understand quaternion representation of crystal orientations
2. See how cubic symmetry operations work
3. Learn the disorientation calculation process
4. Identify CSL (Coincidence Site Lattice) boundaries
5. Visualize the fundamental zone concept

---

In [3]:
# Setup: Import required modules
import sys
from pathlib import Path
import numpy as np

# Add parent's src to path
src_path = Path.cwd().parent / 'MISORI' / 'src'
sys.path.insert(0, str(src_path))

print(f"Current directory: {Path.cwd()}")
print(f"Looking for src at: {src_path}")
print(f"src exists: {src_path.exists()}")

from quaternions import Quaternion, euler_to_quaternion, axis_angle_to_quaternion, quaternion_to_euler
from symmetry import CubicSymmetry
from csl import CSLIdentifier
from misorientation import analyze_axis_angle, analyze_euler_angles

print("✓ Modules loaded successfully")

Current directory: /Users/oq50iqeq/Desktop/PROJECTS/DEVEL/TOOL-4-MISORIENTATION/MISORI
Looking for src at: /Users/oq50iqeq/Desktop/PROJECTS/DEVEL/TOOL-4-MISORIENTATION/MISORI/src
src exists: True
✓ Modules loaded successfully


---
## Part 1: Understanding Quaternions

### What Are Quaternions?

Quaternions are mathematical objects with **4 components**: q = [q₁, q₂, q₃, q₄]

They were invented in 1843 by William Rowan Hamilton to represent 3D rotations. While they may seem more complicated than Euler angles (which only have 3 numbers), quaternions solve critical problems that plague other rotation representations.

Quaternions are an extension of complex numbers used to represent 3D rotations in a compact and numerically stable way.

A quaternion is defined as:

\[
q = w + x\,i + y\,j + z\,k
\]

where:
- \( w \) is the **scalar part**
- \( (x, y, z) \) is the **vector part**

and \( i, j, k \) are imaginary units that follow the rules:

\[
i^2 = j^2 = k^2 = ijk = -1
\]

They avoid gimbal lock (unlike Euler angles) and are more efficient than rotation matrices for interpolation (slerp).


### Unit Quaternions for Rotations

When representing rotations, we use **unit quaternions** with the constraint:

**q₁² + q₂² + q₃² + q₄² = 1**

This means the quaternion lives on the surface of a 4-dimensional sphere. This seemingly abstract property gives quaternions their power.

### Why "No Singularities" Matters

**The Problem with Euler Angles:**

Euler angles (like φ₁, Φ, φ₂) suffer from **gimbal lock** - a mathematical singularity where:
- Two rotation axes become parallel
- You lose one degree of freedom
- Mathematical operations break down (derivatives → infinity, division by zero)
- Interpolation between orientations fails

**Real-world consequences:**
- Spacecraft attitude control systems can lock up
- Animation software produces jerky rotations
- Optimization algorithms crash
- Code needs special cases and error handling


In [4]:
import numpy as np

def quat_from_axis_angle(axis, angle_deg):
    """Return quaternion [w, x, y, z] from axis (3,) and angle in degrees."""
    angle = np.deg2rad(angle_deg)
    axis = np.asarray(axis) / np.linalg.norm(axis)
    w = np.cos(angle / 2)
    x, y, z = axis * np.sin(angle / 2)
    return np.array([w, x, y, z])

# Example: 90° rotation around z-axis
q = quat_from_axis_angle([0, 0, 1], 90)
q

array([0.70710678, 0.        , 0.        , 0.70710678])

In [5]:
# Create a 90° rotation around the z-axis [0,0,1]
angle = 90.0  # degrees
axis = [0, 0, 1]

q = axis_angle_to_quaternion(angle, axis, degrees=True)

print(f"Rotation: {angle}° around {axis}")
print(f"Quaternion: q = [{q.q[0]:.4f}, {q.q[1]:.4f}, {q.q[2]:.4f}, {q.q[3]:.4f}]")
print(f"\nVerification:")
print(f"  Norm = {np.linalg.norm(q.q):.6f} (should be 1.0)")

# Convert back to axis-angle
angle_back, axis_back = q.to_axis_angle()
print(f"\nConverted back:")
print(f"  Angle: {angle_back:.2f}°")
print(f"  Axis: [{axis_back[0]:.4f}, {axis_back[1]:.4f}, {axis_back[2]:.4f}]")

Rotation: 90.0° around [0, 0, 1]
Quaternion: q = [0.0000, 0.0000, 0.7071, 0.7071]

Verification:
  Norm = 1.000000 (should be 1.0)

Converted back:
  Angle: 90.00°
  Axis: [0.0000, 0.0000, 1.0000]


## Quaternion multiplication
If \( q_1 \) and \( q_2 \) are two rotation quaternions,  
then the combined rotation is:

\[
q = q_2 \otimes q_1
\]

Note: Quaternion multiplication is **not commutative**.

In [6]:
def quat_multiply(q2, q1):
    """Hamilton product q = q2 ⊗ q1"""
    w1, x1, y1, z1 = q1
    w2, x2, y2, z2 = q2
    w = w2*w1 - x2*x1 - y2*y1 - z2*z1
    x = w2*x1 + x2*w1 + y2*z1 - z2*y1
    y = w2*y1 - x2*z1 + y2*w1 + z2*x1
    z = w2*z1 + x2*y1 - y2*x1 + z2*w1
    return np.array([w, x, y, z])

# Example: Rotate 90° around z, then 90° around x
qz = quat_from_axis_angle([0, 0, 1], 90)
qx = quat_from_axis_angle([1, 0, 0], 90)
q_combined = quat_multiply(qx, qz)
q_combined

array([ 0.5,  0.5, -0.5,  0.5])

## Rotate a vector using Quaternions:
To rotate a vector **v** by quaternion **q**:

\[
v' = q \, v \, q^{-1}
\]

where the vector \( v \) is treated as a pure quaternion \( [0, v_x, v_y, v_z] \).

In [7]:
def quat_conjugate(q):
    w, x, y, z = q
    return np.array([w, -x, -y, -z])

def quat_rotate_vector(q, v):
    """Rotate 3D vector v by quaternion q."""
    qv = np.concatenate([[0.0], v])
    q_inv = quat_conjugate(q) / np.dot(q, q)
    return quat_multiply(quat_multiply(q, qv), q_inv)[1:]

# Example: Rotate (1, 0, 0) by 90° around z
v = np.array([1, 0, 0])
v_rot = quat_rotate_vector(qz, v)
v_rot

array([2.22044605e-16, 1.00000000e+00, 0.00000000e+00])

## Convert Quaternion → Rotation Matrix

In [8]:
def quat_to_matrix(q):
    """Return 3x3 rotation matrix corresponding to quaternion q."""
    w, x, y, z = q
    return np.array([
        [1 - 2*(y**2 + z**2),   2*(x*y - z*w),     2*(x*z + y*w)],
        [2*(x*y + z*w),         1 - 2*(x**2 + z**2), 2*(y*z - x*w)],
        [2*(x*z - y*w),         2*(y*z + x*w),     1 - 2*(x**2 + y**2)]
    ])

# Check equivalence with numpy's rotation
R = quat_to_matrix(qz)
R @ np.array([1,0,0])

array([2.22044605e-16, 1.00000000e+00, 0.00000000e+00])

### Example 2: Creating Quaternions from Euler Angles

Euler angles (Bunge convention): φ₁, Φ, φ₂
- φ₁: Rotation about Z-axis
- Φ: Rotation about new X-axis
- φ₂: Rotation about new Z-axis

In [9]:
# Euler angles (Bunge convention)
phi1, phi, phi2 = 45.0, 90.0, 30.0

q_euler = euler_to_quaternion(phi1, phi, phi2, degrees=True)

print(f"Euler angles: φ₁={phi1}°, Φ={phi}°, φ₂={phi2}°")
print(f"Quaternion: q = [{q_euler.q[0]:.4f}, {q_euler.q[1]:.4f}, {q_euler.q[2]:.4f}, {q_euler.q[3]:.4f}]")

# Convert to axis-angle
angle_equiv, axis_equiv = q_euler.to_axis_angle()
print(f"\nEquivalent axis-angle:")
print(f"  Angle: {angle_equiv:.2f}°")
print(f"  Axis: [{axis_equiv[0]:.4f}, {axis_equiv[1]:.4f}, {axis_equiv[2]:.4f}]")

# Convert back to Euler angles
phi1_back, phi_back, phi2_back = quaternion_to_euler(q_euler)
print(f"\nConverted back to Euler:")
print(f"  φ₁={phi1_back:.2f}°, Φ={phi_back:.2f}°, φ₂={phi2_back:.2f}°")

Euler angles: φ₁=45.0°, Φ=90.0°, φ₂=30.0°
Quaternion: q = [0.7011, 0.0923, 0.4305, 0.5610]

Equivalent axis-angle:
  Angle: 111.75°
  Axis: [0.8469, 0.1115, 0.5200]

Converted back to Euler:
  φ₁=45.00°, Φ=90.00°, φ₂=30.00°


---
## Part 2: Cubic Crystal Symmetry

Cubic crystals have **24 proper rotation symmetries**:
- 1 identity (0° rotation)
- 6 face rotations (90°, 180°, 270° around <100>)
- 8 body diagonal rotations (120°, 240° around <111>)
- 9 edge rotations (180° around <110>)

Let's examine all 24 symmetry operators:

In [10]:
# Load cubic symmetry operators
symm = CubicSymmetry()

print(f"Total symmetry operators: {symm.num_symm}\n")
print(f"{'Index':<6} {'q1':<8} {'q2':<8} {'q3':<8} {'q4':<8} {'Angle':<8} {'Axis'}")
print("-" * 70)

for i, op in enumerate(symm.operators):
    angle, axis = op.to_axis_angle()
    print(f"{i:<6} {op.q[0]:>7.4f} {op.q[1]:>7.4f} {op.q[2]:>7.4f} {op.q[3]:>7.4f} "
          f"{angle:>7.1f}° [{axis[0]:>5.2f},{axis[1]:>5.2f},{axis[2]:>5.2f}]")

Total symmetry operators: 24

Index  q1       q2       q3       q4       Angle    Axis
----------------------------------------------------------------------
0       0.0000  0.0000  0.0000  1.0000     0.0° [ 0.00, 0.00, 1.00]
1       1.0000  0.0000  0.0000  0.0000   180.0° [ 1.00, 0.00, 0.00]
2       0.0000  1.0000  0.0000  0.0000   180.0° [ 0.00, 1.00, 0.00]
3       0.0000  0.0000  1.0000  0.0000   180.0° [ 0.00, 0.00, 1.00]
4       0.7071  0.0000  0.0000  0.7071    90.0° [ 1.00, 0.00, 0.00]
5       0.0000  0.7071  0.0000  0.7071    90.0° [ 0.00, 1.00, 0.00]
6       0.0000  0.0000  0.7071  0.7071    90.0° [ 0.00, 0.00, 1.00]
7      -0.7071  0.0000  0.0000  0.7071    90.0° [-1.00, 0.00, 0.00]
8       0.0000 -0.7071  0.0000  0.7071    90.0° [ 0.00,-1.00, 0.00]
9       0.0000  0.0000 -0.7071  0.7071    90.0° [ 0.00, 0.00,-1.00]
10      0.7071  0.7071  0.0000  0.0000   180.0° [ 0.71, 0.71, 0.00]
11     -0.7071  0.7071  0.0000  0.0000   180.0° [-0.71, 0.71, 0.00]
12      0.0000  0.7071  0.

### Applying Symmetry Operations

Let's see what happens when we apply different symmetry operators to an orientation:

In [11]:
# Start with a simple orientation: 30° around [1,0,0]
q_test = axis_angle_to_quaternion(30.0, [1, 0, 0])

print(f"Original orientation: 30° around [1,0,0]")
print(f"Quaternion: [{q_test.q[0]:.4f}, {q_test.q[1]:.4f}, {q_test.q[2]:.4f}, {q_test.q[3]:.4f}]\n")

print(f"Applying first 5 symmetry operations:\n")
print(f"{'Symm':<6} {'Result Quaternion':<40} {'Angle':<8} {'Axis'}")
print("-" * 80)

for i in range(5):
    q_result = symm.apply_pre_symmetry(q_test, i)
    angle, axis = q_result.to_axis_angle()
    print(f"{i:<6} [{q_result.q[0]:>6.3f},{q_result.q[1]:>6.3f},{q_result.q[2]:>6.3f},{q_result.q[3]:>6.3f}] "
          f"{angle:>7.1f}° [{axis[0]:>5.2f},{axis[1]:>5.2f},{axis[2]:>5.2f}]")

print("\n→ Different symmetry operators produce symmetrically equivalent orientations")

Original orientation: 30° around [1,0,0]
Quaternion: [0.2588, 0.0000, 0.0000, 0.9659]

Applying first 5 symmetry operations:

Symm   Result Quaternion                        Angle    Axis
--------------------------------------------------------------------------------
0      [ 0.259, 0.000, 0.000, 0.966]    30.0° [ 1.00, 0.00, 0.00]
1      [ 0.966, 0.000, 0.000,-0.259]   210.0° [ 1.00, 0.00, 0.00]
2      [ 0.000, 0.966,-0.259, 0.000]   180.0° [ 0.00, 0.97,-0.26]
3      [ 0.000, 0.259, 0.966, 0.000]   180.0° [ 0.00, 0.26, 0.97]
4      [ 0.866, 0.000, 0.000, 0.500]   120.0° [ 1.00, 0.00, 0.00]

→ Different symmetry operators produce symmetrically equivalent orientations


---
## Part 3: Calculating Misorientation

**Misorientation** between two grains: Q_mis = Q₁ · Q₂⁻¹

Let's calculate the misorientation between two grains step by step:

In [16]:
# Define two grain orientations
q1 = axis_angle_to_quaternion(0, [1, 0, 0])      # Grain 1: identity (reference)
q2 = axis_angle_to_quaternion(57, [1, 1, 1])    # Grain 2: 60° around [111]

print("Grain 1 (reference):")
print(f"  Quaternion: [{q1.q[0]:.4f}, {q1.q[1]:.4f}, {q1.q[2]:.4f}, {q1.q[3]:.4f}]")

print("\nGrain 2:")
print(f"  Input: 50° around [1,1,1]")
print(f"  Quaternion: [{q2.q[0]:.4f}, {q2.q[1]:.4f}, {q2.q[2]:.4f}, {q2.q[3]:.4f}]")

# Calculate misorientation Q1 * Q2^-1
q_mis = q1.misorientation(q2)

print("\nMisorientation (Q₁ · Q₂⁻¹):")
print(f"  Quaternion: [{q_mis.q[0]:.4f}, {q_mis.q[1]:.4f}, {q_mis.q[2]:.4f}, {q_mis.q[3]:.4f}]")

angle_mis, axis_mis = q_mis.to_axis_angle()
print(f"  Angle: {angle_mis:.2f}°")
print(f"  Axis: [{axis_mis[0]:.4f}, {axis_mis[1]:.4f}, {axis_mis[2]:.4f}]")

Grain 1 (reference):
  Quaternion: [0.0000, 0.0000, 0.0000, 1.0000]

Grain 2:
  Input: 50° around [1,1,1]
  Quaternion: [0.2755, 0.2755, 0.2755, 0.8788]

Misorientation (Q₁ · Q₂⁻¹):
  Quaternion: [-0.2755, -0.2755, -0.2755, 0.8788]
  Angle: 57.00°
  Axis: [-0.5774, -0.5774, -0.5774]


---
## Part 4: Finding the Disorientation

**Disorientation** = Minimum angle representation after applying all symmetry operations

We need to check 24 × 24 = **576 combinations** of symmetry operators to find the smallest angle!

### The Fundamental Zone

For cubic crystals, the disorientation must satisfy: **0 ≤ q₁ ≤ q₂ ≤ q₃ ≤ q₄**

Let's search through all combinations:

In [17]:
# Continue with the misorientation from above
print("Searching through all 576 symmetry combinations...\n")

# Initialize symmetry object
symm = CubicSymmetry()  

# Track the search process
candidates = []
qmax = 0.0

for i in range(24):
    quint = symm.apply_pre_symmetry(q_mis, i)
    
    for j in range(24):
        quintn = symm.apply_post_symmetry(quint, j)
        
        # Try both positive and negative
        for neg_all in [False, True]:
            qqn = Quaternion(-quintn.q) if neg_all else quintn
            
            for neg_q4 in [False, True]:
                qresult = qqn.q.copy()
                if neg_q4:
                    qresult[3] = -qqn.q[3]
                
                # Check fundamental zone: 0 ≤ q1 ≤ q2 ≤ q3 ≤ q4
                if (qresult[0] >= 0.0 and 
                    qresult[0] <= qresult[1] and
                    qresult[1] <= qresult[2] and
                    qresult[2] <= qresult[3]):
                    
                    candidates.append({
                        'i': i, 'j': j,
                        'q': qresult.copy(),
                        'q4': qresult[3]
                    })
                    
                    if qresult[3] > qmax:
                        qmax = qresult[3]

print(f"Found {len(candidates)} valid candidates in fundamental zone")
print(f"\nTop 10 candidates (sorted by q₄, largest angle → smallest):")
print(f"\n{'Symm(i,j)':<12} {'Quaternion [q1, q2, q3, q4]':<45} {'Angle'}")
print("-" * 75)

# Sort by q4 (descending) and show top 10
candidates_sorted = sorted(candidates, key=lambda x: x['q4'], reverse=True)
for k, c in enumerate(candidates_sorted[:10]):
    q_temp = Quaternion(c['q'])
    angle_temp, _ = q_temp.to_axis_angle()
    print(f"({c['i']:>2},{c['j']:>2})      "
          f"[{c['q'][0]:>6.3f}, {c['q'][1]:>6.3f}, {c['q'][2]:>6.3f}, {c['q'][3]:>6.3f}]  "
          f"{angle_temp:>6.2f}°")

print(f"\n→ The winner (smallest angle) is at the top!")

Searching through all 576 symmetry combinations...

Found 22 valid candidates in fundamental zone

Top 10 candidates (sorted by q₄, largest angle → smallest):

Symm(i,j)    Quaternion [q1, q2, q3, q4]                   Angle
---------------------------------------------------------------------------
(11,11)      [ 0.275,  0.275,  0.275,  0.879]   57.00°
(13,13)      [ 0.275,  0.275,  0.275,  0.879]   57.00°
(15,15)      [ 0.275,  0.275,  0.275,  0.879]   57.00°
(16,16)      [ 0.275,  0.275,  0.275,  0.879]   57.00°
(17,17)      [ 0.275,  0.275,  0.275,  0.879]   57.00°
( 0, 0)      [ 0.275,  0.275,  0.275,  0.879]   57.00°
(11,15)      [ 0.302,  0.302,  0.302,  0.853]   63.00°
(13,11)      [ 0.302,  0.302,  0.302,  0.853]   63.00°
(15,13)      [ 0.302,  0.302,  0.302,  0.853]   63.00°
( 0,17)      [ 0.302,  0.302,  0.302,  0.853]   63.00°

→ The winner (smallest angle) is at the top!


### Final Disorientation Result

In [18]:
# Use the built-in function to get the disorientation
disor, idx1, idx2, neg_q4, neg_all = symm.find_disorientation(q_mis)

angle_disor, axis_disor = disor.to_axis_angle()

print("DISORIENTATION RESULT")
print("=" * 60)
print(f"Quaternion: [{disor.q[0]:.4f}, {disor.q[1]:.4f}, {disor.q[2]:.4f}, {disor.q[3]:.4f}]")
print(f"Angle: {angle_disor:.2f}°")
print(f"Axis: [{axis_disor[0]:.4f}, {axis_disor[1]:.4f}, {axis_disor[2]:.4f}]")
print(f"\nSymmetry operators used: ({idx1}, {idx2})")
print(f"Negations applied: q4={neg_q4}, all={neg_all}")
print("\n→ This is the MINIMUM angle representation in the fundamental zone")

DISORIENTATION RESULT
Quaternion: [0.2755, 0.2755, 0.2755, 0.8788]
Angle: 57.00°
Axis: [0.5774, 0.5774, 0.5774]

Symmetry operators used: (11, 11)
Negations applied: q4=False, all=False

→ This is the MINIMUM angle representation in the fundamental zone


---
## Part 5: CSL Boundary Identification

**CSL (Coincidence Site Lattice)** boundaries are special orientations where a fraction of lattice sites coincide.

**Σ value** = reciprocal density of coincident sites (low Σ GBs can have special properties)

**Brandon criterion**: Δθ_max = 15° / √Σ (allowable deviation from exact CSL)

### Example: Famous Σ3  Boundary

In [19]:
# The famous 60° [111]  boundary
csl = CSLIdentifier()

#q2 = axis_angle_to_quaternion(60, [1, 1, 1]) 

# Identify based on our disorientation
#angle_disor = 60
#disor = q2
sigma, deviation = csl.identify_boundary(disor, angle_disor)


print("CSL IDENTIFICATION")
print("=" * 60)
print(f"Disorientation: {angle_disor:.2f}° around [{axis_disor[0]:.3f}, {axis_disor[1]:.3f}, {axis_disor[2]:.3f}]")
print(f"\nIdentified as: Σ{sigma}")
print(f"Deviation: {deviation:.3f}")

if sigma:
    brandon = csl.brandon_criterion(sigma)
    print(f"Brandon criterion for Σ{sigma}: ±{brandon:.2f}°")
    
    if deviation < 1.0:
        print(f"\n✓ TRUE CSL boundary (within Brandon criterion)")
    else:
        print(f"\n⚠ Near-CSL (outside Brandon criterion)")

print("\n" + "=" * 60)
print("If the GB plane would be {111} this would be the famous coherent twin boundary!")
print("One of the most common and important special boundaries.")

DEBUG Σ19:
  delta_q = [0.051406, 0.051406, 0.051406, 0.996029]
  q4_clamped = 0.996029
  theta = 10.2158°
  Brandon = 3.4412°
FINAL: best_sigma=3, min_criterion=0.346097
CSL IDENTIFICATION
Disorientation: 57.00° around [0.577, 0.577, 0.577]

Identified as: Σ3
Deviation: 0.346
Brandon criterion for Σ3: ±8.66°

✓ TRUE CSL boundary (within Brandon criterion)

If the GB plane would be {111} this would be the famous coherent twin boundary!
One of the most common and important special boundaries.


In [19]:
disor

Quaternion([0.2440, 0.2440, 0.2440, 0.9063])

### All Available CSL Boundaries

Let's see what CSL boundaries are in the database:

In [10]:
print(f"Available CSL boundaries (Σ3 to Σ35):\n")
print(f"{'Σ':<4} {'Angle':<8} {'Axis':<10} {'Brandon':<10} {'Quaternion'}")
print("-" * 70)

for csl_boundary in csl.boundaries:
    brandon = csl.brandon_criterion(csl_boundary.sigma)
    uvw_str = f"[{csl_boundary.uvw[0]}{csl_boundary.uvw[1]}{csl_boundary.uvw[2]}]"
    q = csl_boundary.quaternion.q
    
    print(f"{csl_boundary.sigma:<4} {csl_boundary.theta:<8.1f} {uvw_str:<10} "
          f"±{brandon:<8.2f} [{q[0]:.3f},{q[1]:.3f},{q[2]:.3f},{q[3]:.3f}]")

print(f"\nTotal: {len(csl.boundaries)} CSL boundary types")

Available CSL boundaries (Σ3 to Σ35):

Σ    Angle    Axis       Brandon    Quaternion
----------------------------------------------------------------------
3    60.0     [111]      ±8.66     [0.289,0.289,0.289,0.866]
5    36.9     [100]      ±6.71     [0.000,0.000,0.316,0.949]
7    38.2     [111]      ±5.67     [0.188,0.188,0.188,0.944]
9    38.9     [110]      ±5.00     [0.000,0.236,0.236,0.943]
11   50.5     [110]      ±4.52     [0.000,0.302,0.302,0.905]
13   22.6     [100]      ±4.16     [0.000,0.000,0.196,0.981]
13   27.8     [111]      ±4.16     [0.139,0.139,0.139,0.973]
15   48.2     [210]      ±3.87     [0.000,0.186,0.371,0.910]
17   28.1     [100]      ±3.64     [0.000,0.000,0.243,0.970]
17   61.9     [221]      ±3.64     [0.171,0.342,0.342,0.857]
19   26.5     [110]      ±3.44     [0.000,0.162,0.162,0.973]
19   46.8     [111]      ±3.44     [0.229,0.229,0.229,0.919]
21   21.8     [111]      ±3.27     [0.109,0.109,0.109,0.982]
21   44.4     [211]      ±3.27     [0.154,0.154,0.

---
## Part 6: Complete Analysis Examples

Now let's use the high-level API to analyze several famous boundaries:

### Test Case 1: Σ3  (60° [111])

In [14]:
gb1 = analyze_axis_angle(60.0, [1, 1, 1])

print("="*70)
print("TEST CASE 1: Σ3 BOUNDARY")
print("="*70)
print(f"Input: 60° around [1,1,1]")
print(f"\nResults:")
print(f"  Disorientation angle: {gb1.angle:.2f}°")
print(f"  Axis: [{gb1.axis[0]:.4f}, {gb1.axis[1]:.4f}, {gb1.axis[2]:.4f}]")
print(f"  Quaternion: [{gb1.disorientation.q[0]:.4f}, {gb1.disorientation.q[1]:.4f}, "
      f"{gb1.disorientation.q[2]:.4f}, {gb1.disorientation.q[3]:.4f}]")
print(f"  Σ value: {gb1.sigma}")
print(f"  Deviation: {gb1.deviation:.3f}")
print(f"  Classification: {gb1._classify_boundary()}")
print(f"\n→ This is the most common twin boundary in FCC metals!")

TEST CASE 1: Σ3 BOUNDARY
Input: 60° around [1,1,1]

Results:
  Disorientation angle: 60.00°
  Axis: [0.5774, 0.5774, 0.5774]
  Quaternion: [0.2887, 0.2887, 0.2887, 0.8660]
  Σ value: 3
  Deviation: 0.000
  Classification: CSL Σ3

→ This is the most common twin boundary in FCC metals!


### Test Case 2: Σ5 Boundary (36.9° [100])

In [11]:
gb2 = analyze_axis_angle(36.9, [1, 0, 0])

print("="*70)
print("TEST CASE 2: Σ5 BOUNDARY")
print("="*70)
print(f"Input: 36.9° around [1,0,0]")
print(f"\nResults:")
print(f"  Disorientation angle: {gb2.angle:.2f}°")
print(f"  Axis: [{gb2.axis[0]:.4f}, {gb2.axis[1]:.4f}, {gb2.axis[2]:.4f}]")
print(f"  Σ value: {gb2.sigma}")
print(f"  Deviation: {gb2.deviation:.3f}")
print(f"  Classification: {gb2._classify_boundary()}")
print(f"\n→ Common in annealed copper and other FCC metals")

TEST CASE 2: Σ5 BOUNDARY
Input: 36.9° around [1,0,0]

Results:
  Disorientation angle: 36.90°
  Axis: [0.0000, 0.0000, 1.0000]
  Σ value: 5
  Deviation: 0.000
  Classification: CSL Σ5

→ Common in annealed copper and other FCC metals


### Test Case 3: Low-Angle Boundary

In [16]:
gb3 = analyze_axis_angle(5.0, [1, 0, 0])

print("="*70)
print("TEST CASE 3: LOW-ANGLE BOUNDARY")
print("="*70)
print(f"Input: 5.0° around [1,0,0]")
print(f"\nResults:")
print(f"  Disorientation angle: {gb3.angle:.2f}°")
print(f"  Axis: [{gb3.axis[0]:.4f}, {gb3.axis[1]:.4f}, {gb3.axis[2]:.4f}]")
print(f"  Σ value: {gb3.sigma}")
print(f"  Classification: {gb3._classify_boundary()}")
print(f"\n→ Low-angle boundaries (θ < 15°) are treated as Σ1")
print(f"→ They consist of arrays of dislocations")

TEST CASE 3: LOW-ANGLE BOUNDARY
Input: 5.0° around [1,0,0]

Results:
  Disorientation angle: 5.00°
  Axis: [0.0000, 0.0000, 1.0000]
  Σ value: 1
  Classification: Low-angle

→ Low-angle boundaries (θ < 15°) are treated as Σ1
→ They consist of arrays of dislocations


### Test Case 4: General High-Angle Boundary

In [17]:
gb4 = analyze_axis_angle(42.5, [1, 2, 3])

print("="*70)
print("TEST CASE 4: GENERAL HIGH-ANGLE BOUNDARY")
print("="*70)
print(f"Input: 42.5° around [1,2,3]")
print(f"\nResults:")
print(f"  Disorientation angle: {gb4.angle:.2f}°")
print(f"  Axis: [{gb4.axis[0]:.4f}, {gb4.axis[1]:.4f}, {gb4.axis[2]:.4f}]")
print(f"  Σ value: {gb4.sigma}")
print(f"  Deviation: {gb4.deviation:.3f}" if gb4.sigma else "  N/A")
print(f"  Classification: {gb4._classify_boundary()}")
print(f"\n→ Random orientations typically don't match CSL boundaries")
print(f"→ If Σ is assigned, check if deviation < 1.0 (within Brandon)")

TEST CASE 4: GENERAL HIGH-ANGLE BOUNDARY
Input: 42.5° around [1,2,3]

Results:
  Disorientation angle: 42.50°
  Axis: [0.2673, 0.5345, 0.8018]
  Σ value: 29
  Deviation: 1.823
  Classification: Near-CSL Σ29

→ Random orientations typically don't match CSL boundaries
→ If Σ is assigned, check if deviation < 1.0 (within Brandon)


---
## Part 7: Euler Angle Input

In real experiments, orientations are often measured as Euler angles from EBSD.

Let's analyze a boundary from two Euler angle measurements:

In [18]:
# Two hypothetical grains from EBSD measurement
phi1_a, phi_a, phi2_a = 0.0, 0.0, 0.0      # Grain A (reference)
phi1_b, phi_b, phi2_b = 45.0, 90.0, 45.0   # Grain B

gb_euler = analyze_euler_angles(phi1_a, phi_a, phi2_a, phi1_b, phi_b, phi2_b)

print("="*70)
print("ANALYSIS FROM EULER ANGLES")
print("="*70)
print(f"Grain A: φ₁={phi1_a}°, Φ={phi_a}°, φ₂={phi2_a}°")
print(f"Grain B: φ₁={phi1_b}°, Φ={phi_b}°, φ₂={phi2_b}°")
print(f"\nResults:")
print(f"  Disorientation angle: {gb_euler.angle:.2f}°")
print(f"  Axis: [{gb_euler.axis[0]:.4f}, {gb_euler.axis[1]:.4f}, {gb_euler.axis[2]:.4f}]")
print(f"  Quaternion: [{gb_euler.disorientation.q[0]:.4f}, {gb_euler.disorientation.q[1]:.4f}, "
      f"{gb_euler.disorientation.q[2]:.4f}, {gb_euler.disorientation.q[3]:.4f}]")
print(f"  Σ value: {gb_euler.sigma}")
print(f"  Deviation: {gb_euler.deviation:.3f}")
print(f"  Classification: {gb_euler._classify_boundary()}")

ANALYSIS FROM EULER ANGLES
Grain A: φ₁=0.0°, Φ=0.0°, φ₂=0.0°
Grain B: φ₁=45.0°, Φ=90.0°, φ₂=45.0°

Results:
  Disorientation angle: 62.80°
  Axis: [0.2811, 0.6786, 0.6786]
  Quaternion: [0.1464, 0.3536, 0.3536, 0.8536]
  Σ value: 17
  Deviation: 1.540
  Classification: Near-CSL Σ17


---
## Summary: Key Concepts

### What We Learned:

1. **Quaternions** provide a singularity-free way to represent rotations
   - 4 components: [q₁, q₂, q₃, q₄]
   - Relationship to axis-angle: q = [n·sin(θ/2), cos(θ/2)]

2. **Cubic Symmetry** has 24 proper rotations
   - Must check all 24×24 = 576 combinations
   - Goal: find minimum angle representation

3. **Fundamental Zone** for cubic crystals:
   - Constraint: 0 ≤ q₁ ≤ q₂ ≤ q₃ ≤ q₄
   - Ensures unique representation

4. **CSL Boundaries** are special misorientations:
   - Σ3 (60°/[111]): coherent twin
   - Σ5 (36.9°/[100]): common in FCC
   - Brandon criterion: Δθ_max = 15°/√Σ

5. **Classification**:
   - Low-angle (θ < 15°): Σ1, dislocation arrays
   - CSL (deviation < 1.0): special boundaries
   - General: random high-angle

### Practical Applications:

- **Materials characterization**: EBSD texture analysis
- **Grain boundary engineering**: Optimize properties
- **Recrystallization studies**: Track orientation changes
- **Interface modeling**: Predict boundary properties

---

## Exercises

Try these on your own:

1. Calculate the disorientation for Σ7 boundary: 38.2° around [111]
2. Find a general boundary and verify it's not CSL
3. Input your own EBSD data as Euler angles
4. Compare results with experimental grain boundary energy data

---

In [17]:
from misorientation import analyze_axis_angle
gb = analyze_axis_angle(60, [1,1,1])
print(f"Sigma: {gb.sigma}, Deviation: {gb.deviation:.6f}")

Sigma: 3, Deviation: 0.460701


In [18]:
# Add this debug cell to your notebook to see what's happening:
from csl import CSLIdentifier
from quaternions import axis_angle_to_quaternion

csl = CSLIdentifier()
q_input = axis_angle_to_quaternion(60, [1,1,1])

print("Input quaternion:", q_input.q)
print("\nStored CSL Σ3 quaternions:")
for boundary in csl.boundaries:
    if boundary.sigma == 3:
        print(f"  {boundary.quaternion.q}")
        dot_product = np.dot(q_input.q, boundary.quaternion.q)
        print(f"  Dot product: {dot_product}")
        angle = 2.0 * np.arccos(np.clip(np.abs(dot_product), 0.0, 1.0)) * 180.0 / np.pi
        print(f"  Angle difference: {angle:.4f}°")

Input quaternion: [0.28867513 0.28867513 0.28867513 0.8660254 ]

Stored CSL Σ3 quaternions:
  [0.288 0.288 0.288 0.866]
  Dot product: 0.9993933159672422
  Angle difference: 3.9918°


In [1]:
from misorientation import analyze_axis_angle
gb = analyze_axis_angle(60, [1,1,1])
print(f"Sigma: {gb.sigma}, Deviation: {gb.deviation:.6f}")

Sigma: 3, Deviation: 0.000000


In [2]:
gb = analyze_axis_angle(50, [1, 1, 1])
print(f"Angle: {gb.angle:.2f}°")
print(f"Axis: [{gb.axis[0]:.3f}, {gb.axis[1]:.3f}, {gb.axis[2]:.3f}]")
print(f"Σ: {gb.sigma}")
print(f"Deviation: {gb.deviation:.3f}")

Angle: 50.00°
Axis: [0.577, 0.577, 0.577]
Σ: 19
Deviation: 0.000


In [3]:
gb = analyze_axis_angle(50, [1, 1, 1])
print(f"Angle: {gb.angle:.2f}°")
print(f"Axis: [{gb.axis[0]:.3f}, {gb.axis[1]:.3f}, {gb.axis[2]:.3f}]")
print(f"Quaternion: [{gb.disorientation.q[0]:.6f}, {gb.disorientation.q[1]:.6f}, {gb.disorientation.q[2]:.6f}, {gb.disorientation.q[3]:.6f}]")
print(f"Σ: {gb.sigma}")
print(f"Deviation: {gb.deviation:.6f}")

# Now let's check the Σ19 CSL boundary data:
from src.csl import CSLIdentifier
csl_id = CSLIdentifier()
sigma19_boundaries = csl_id.get_boundary_info(19)
for boundary in sigma19_boundaries:
    print(f"\nΣ19: {boundary.theta:.2f}° around {boundary.uvw}")
    print(f"CSL Quat: [{boundary.quaternion.q[0]:.6f}, {boundary.quaternion.q[1]:.6f}, {boundary.quaternion.q[2]:.6f}, {boundary.quaternion.q[3]:.6f}]")

Angle: 50.00°
Axis: [0.577, 0.577, 0.577]
Quaternion: [0.243999, 0.243999, 0.243999, 0.906308]
Σ: 19
Deviation: 0.000000

Σ19: 26.53° around [1, 1, 0]
CSL Quat: [0.000000, 0.162221, 0.162221, 0.973249]

Σ19: 46.83° around [1, 1, 1]
CSL Quat: [0.229403, 0.229403, 0.229403, 0.918559]


In [1]:
from misorientation import analyze_axis_angle
gb = analyze_axis_angle(57, [1, 1, 1])
print(f"Deviation: {gb.deviation:.6f}")

DEBUG Σ19:
  delta_q = [0.051406, 0.051406, 0.051406, 0.996029]
  q4_clamped = 0.996029
  theta = 10.2158°
  Brandon = 3.4412°
FINAL: best_sigma=3, min_criterion=0.346097
Deviation: 0.346097
